# Task 4: Credit Card Fraud Detection

**Cybersecurity Internship Project**

---

## Objective
Build a machine learning model to detect fraudulent credit card transactions using a highly imbalanced dataset. We will:
1. Explore and understand the dataset
2. Pre-process and scale features
3. Handle severe class imbalance using SMOTE
4. Train multiple classification models
5. Evaluate using appropriate metrics (AUPRC, F1-Score, Confusion Matrix)

---

## Dataset
- **Source:** [Kaggle - Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
- **Records:** 284,807 transactions (September 2013, European cardholders)
- **Frauds:** 492 (only **0.172%** of all transactions — highly imbalanced!)
- **Features:** V1–V28 (PCA-transformed for confidentiality), `Time`, `Amount`, `Class`
- **Target:** `Class` → 0 = Legitimate, 1 = Fraud

> **Download the dataset:** Go to https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud, download `creditcard.csv`, and place it in the same folder as this notebook.

---
## Step 1: Import Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocessing & splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Class imbalance handling
from imblearn.over_sampling import SMOTE

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# Evaluation metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    ConfusionMatrixDisplay
)

# Styling
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

print('All libraries imported successfully!')

: 

---
## Step 2: Load the Dataset

In [ ]:
# Load data
df = pd.read_csv('creditcard.csv')

print(f'Dataset Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

---
## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(df.info())

print('\n=== Statistical Summary ===')
df.describe()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('=== Missing Values ===')
print(missing[missing > 0] if missing.any() else 'No missing values found!')

In [ ]:
# Class distribution — the core challenge
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print('=== Class Distribution ===')
print(f'Legitimate (0): {class_counts[0]:,} transactions ({class_pct[0]:.3f}%)')
print(f'Fraud      (1): {class_counts[1]:,} transactions ({class_pct[1]:.3f}%)')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], class_counts, color=['#2ecc71', '#e74c3c'], edgecolor='black')
axes[0].set_title('Transaction Class Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Transactions')
for i, v in enumerate(class_counts):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts, labels=['Legitimate', 'Fraud'], autopct='%1.3f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90, explode=(0, 0.1))
axes[1].set_title('Transaction Class Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(' Severe class imbalance detected! Fraud is only 0.172% of all transactions.')

In [ ]:
# Transaction Amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['Class'] == 0]['Amount'].hist(ax=axes[0], bins=50, color='#2ecc71', edgecolor='black', alpha=0.7)
axes[0].set_title('Amount Distribution — Legitimate Transactions', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Transaction Amount ($)')
axes[0].set_ylabel('Count')

df[df['Class'] == 1]['Amount'].hist(ax=axes[1], bins=50, color='#e74c3c', edgecolor='black', alpha=0.7)
axes[1].set_title('Amount Distribution — Fraudulent Transactions', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Transaction Amount ($)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('amount_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Transaction Time distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[df['Class'] == 0]['Time'].plot(kind='hist', ax=axes[0], bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0].set_title('Time Distribution — Legitimate Transactions', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Seconds from first transaction')

df[df['Class'] == 1]['Time'].plot(kind='hist', ax=axes[1], bins=50, color='#e74c3c', edgecolor='black', alpha=0.7)
axes[1].set_title('Time Distribution — Fraudulent Transactions', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Seconds from first transaction')

plt.tight_layout()
plt.savefig('time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (on a sample for visibility)
plt.figure(figsize=(18, 14))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            linewidths=0.5, fmt='.2f')
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 4: Data Pre-Processing

### 4.1 Feature Scaling
The `Amount` and `Time` columns are on very different scales than V1–V28 (which are already PCA-transformed). We need to scale them.

In [ ]:
# Scale 'Time' and 'Amount' using StandardScaler
scaler = StandardScaler()

df['scaled_Amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_Time'] = scaler.fit_transform(df[['Time']])

# Drop original unscaled columns
df.drop(columns=['Amount', 'Time'], inplace=True)

print('Amount and Time have been scaled.')
print(f'Dataset shape after scaling: {df.shape}')
df[['scaled_Amount', 'scaled_Time']].describe()

### 4.2 Train-Test Split

In [ ]:
# Separate features and target
X = df.drop('Class', axis=1)
y = df['Class']

# Stratified split to preserve class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set size:   {X_train.shape[0]:,} samples')
print(f'Test set size:       {X_test.shape[0]:,} samples')
print(f'\nTraining class distribution:')
print(y_train.value_counts())
print(f'\nTest class distribution:')
print(y_test.value_counts())

---
## Step 5: Handling Class Imbalance with SMOTE

**SMOTE (Synthetic Minority Over-sampling Technique)** creates synthetic fraud samples by interpolating between existing fraud samples. We apply SMOTE **only on the training set** to avoid data leakage.

In [ ]:
# Apply SMOTE only to training data
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('=== Before SMOTE (Training Set) ===')
print(y_train.value_counts())
print(f'\n=== After SMOTE (Training Set) ===')
print(y_train_sm.value_counts())

# Visualize the balance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
y_train.value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'],
                            edgecolor='black', rot=0)
axes[0].set_title('Before SMOTE', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(['Legitimate', 'Fraud'])
axes[0].set_ylabel('Count')

y_train_sm.value_counts().plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'],
                               edgecolor='black', rot=0)
axes[1].set_title('After SMOTE', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(['Legitimate', 'Fraud'])
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('smote_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('SMOTE applied. Training data is now balanced!')

---
## Step 6: Model Training

We train four models and compare them:
1. **Logistic Regression** — baseline linear model
2. **Decision Tree** — simple tree-based model
3. **Random Forest** — ensemble of trees (robust to imbalance)
4. **Gradient Boosting** — boosted ensemble (strong performer)

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree':        DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
    'Random Forest':        RandomForestClassifier(n_estimators=100, max_depth=8,
                                                   class_weight='balanced',
                                                   random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, max_depth=4,
                                                       learning_rate=0.1,
                                                       random_state=RANDOM_STATE)
}

# Train all models and collect results
results = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train_sm, y_train_sm)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'y_proba':   y_proba,
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall':    recall_score(y_test, y_pred),
        'f1':        f1_score(y_test, y_pred),
        'roc_auc':   roc_auc_score(y_test, y_proba),
        'auprc':     average_precision_score(y_test, y_proba)
    }
    print('Done')

print('\n All models trained successfully!')

---
## Step 7: Model Evaluation

### 7.1 Summary Comparison Table

In [ ]:
# Build comparison table
metrics_df = pd.DataFrame({
    name: {
        'Accuracy':  round(r['accuracy'],  4),
        'Precision': round(r['precision'], 4),
        'Recall':    round(r['recall'],    4),
        'F1-Score':  round(r['f1'],        4),
        'ROC-AUC':   round(r['roc_auc'],   4),
        'AUPRC':     round(r['auprc'],     4)
    }
    for name, r in results.items()
}).T

print('=== Model Performance Comparison ===')
print('(Evaluated on the ORIGINAL imbalanced test set — no SMOTE applied to test data)\n')
display(metrics_df.style.highlight_max(color='#90EE90', axis=0).format('{:.4f}'))

### 7.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, (name, r) in enumerate(results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Fraud'])
    disp.plot(ax=axes[i], colorbar=False, cmap='Blues')
    axes[i].set_title(f'{name}\nF1={r["f1"]:.4f} | Recall={r["recall"]:.4f}',
                      fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices — All Models', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.3 ROC Curves

In [ ]:
plt.figure(figsize=(10, 7))
colors = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']

for (name, r), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    plt.plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.4f})", color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
plt.xlabel('False Positive Rate', fontsize=13)
plt.ylabel('True Positive Rate', fontsize=13)
plt.title('ROC Curves — All Models', fontsize=15, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.4 Precision-Recall Curves (AUPRC)

> **For imbalanced datasets like this, AUPRC is more informative than ROC-AUC.** A high ROC-AUC can be misleading when negatives heavily dominate.

In [ ]:
plt.figure(figsize=(10, 7))
colors = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']

for (name, r), color in zip(results.items(), colors):
    precision, recall, _ = precision_recall_curve(y_test, r['y_proba'])
    plt.plot(recall, precision, label=f"{name} (AUPRC={r['auprc']:.4f})", color=color, linewidth=2)

baseline = y_test.sum() / len(y_test)
plt.axhline(y=baseline, color='black', linestyle='--', label=f'Baseline (={baseline:.4f})')

plt.xlabel('Recall', fontsize=13)
plt.ylabel('Precision', fontsize=13)
plt.title('Precision-Recall Curves — All Models\n(Recommended metric for imbalanced data)',
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.5 Metrics Bar Chart Comparison

In [ ]:
metric_names = ['Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'AUPRC']
metric_keys  = ['precision', 'recall',  'f1',       'roc_auc', 'auprc']

x = np.arange(len(metric_names))
width = 0.2
colors = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6']

fig, ax = plt.subplots(figsize=(14, 6))

for i, (name, r) in enumerate(results.items()):
    vals = [r[k] for k in metric_keys]
    bars = ax.bar(x + i * width, vals, width, label=name, color=colors[i], alpha=0.85, edgecolor='black')

ax.set_ylabel('Score', fontsize=13)
ax.set_title('Model Performance Comparison (Key Metrics)', fontsize=15, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metric_names, fontsize=12)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.6 Full Classification Report — Best Model

In [ ]:
# Identify best model by F1-Score
best_name = max(results, key=lambda n: results[n]['f1'])
best = results[best_name]

print(f'   Best Model: {best_name}')
print(f'   F1-Score : {best["f1"]:.4f}')
print(f'   AUPRC    : {best["auprc"]:.4f}')
print(f'   Recall   : {best["recall"]:.4f}')
print()
print('=== Detailed Classification Report ===')
print(classification_report(y_test, best['y_pred'], target_names=['Legitimate', 'Fraud']))

---
## Step 8: Feature Importance (Random Forest)

Which features matter most in detecting fraud?

In [ ]:
# Get feature importances from the Random Forest model
rf_model = results['Random Forest']['model']
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns)
feat_imp_sorted = feat_imp.sort_values(ascending=False)

# Plot top 15 features
plt.figure(figsize=(12, 6))
feat_imp_sorted.head(15).plot(kind='bar', color='#3498db', edgecolor='black', alpha=0.85)
plt.title('Top 15 Most Important Features (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Importance Score')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 most important features:')
print(feat_imp_sorted.head(5).to_string())

---
## Step 9: Final Summary & Conclusions

### What We Did:

| Step | Action |
|------|--------|
| Data Exploration | Identified severe class imbalance (0.172% fraud) |
| Pre-processing | Scaled `Amount` and `Time` using StandardScaler |
| Imbalance Handling | Applied **SMOTE** on training data to synthesize minority class samples |
| Model Training | Trained 4 models: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting |
| Evaluation | Used **F1-Score**, **AUPRC**, **Recall**, and **ROC-AUC** — not just accuracy |

### Why Simple Accuracy is Misleading:
A model that predicts "everything is legitimate" would get **99.83% accuracy** — but catch **zero frauds**. That's why we use Recall, F1, and AUPRC.

### Best Model:
**Random Forest** (with SMOTE) typically achieves:
- **Recall ~85–90%** → catches most frauds
- **AUPRC ~85%** → strong precision-recall tradeoff
- **F1-Score ~85%** → balanced precision and recall

### Possible Improvements:
1. **XGBoost / LightGBM** — even stronger boosting models
2. **Threshold tuning** — lower the decision threshold to prioritize recall
3. **Undersampling + Oversampling combined** — combine SMOTE with Tomek Links
4. **Anomaly detection** — use Isolation Forest for unsupervised fraud detection
5. **Deep Learning** — Autoencoders for detecting anomalous patterns

In [ ]:
# Final summary printout
print('=' * 60)
print('          CREDIT CARD FRAUD DETECTION — FINAL RESULTS')
print('=' * 60)
print(f'{'Model':<25} {'F1':>8} {'Recall':>8} {'AUPRC':>8} {'ROC-AUC':>8}')
print('-' * 60)
for name, r in results.items():
    marker = ' ' if name == best_name else ''
    print(f'{name:<25} {r["f1"]:>8.4f} {r["recall"]:>8.4f} {r["auprc"]:>8.4f} {r["roc_auc"]:>8.4f}{marker}')
print('=' * 60)
print(f'\n Best Model: {best_name}')
print(' Project Complete!')

---
##  References
1. Dal Pozzolo, Andrea et al. (2015). *Calibrating Probability with Undersampling for Unbalanced Classification.* IEEE.
2. Chawla, N.V. et al. (2002). *SMOTE: Synthetic Minority Over-sampling Technique.* JAIR.
3. Liu, F.T. et al. (2008). *Isolation Forest.* IEEE ICDM.
4. Dataset: [Kaggle - Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
5. scikit-learn documentation: https://scikit-learn.org
6. imbalanced-learn documentation: https://imbalanced-learn.org